In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import collections
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Subset

from src.dataset import make_loaders, simulate_imbalance
from src.evaluate import predict_loader, summarize
from src.experiment import run_experiment, save_result
from src.model import CNNBaseline
from src.sampler import BalancedSampler
from src.train import fit, make_scheduler
from src.utils import get_device, set_seed

SEED, EPOCHS, IMG_SIZE, BATCH = 42, 30, 128, 32
DATA = '../data/processed'
RESULTS = '../results/ablations.csv'
device = get_device()



train_ds, _, train_loader, val_loader = make_loaders(DATA, IMG_SIZE, BATCH, SEED)
print(train_ds.class_to_idx)

keep_fraction = {0: 0.2, 4: 0.3}   
keep = simulate_imbalance(train_ds.targets, keep_fraction, SEED)
imb_ds = Subset(train_ds, keep)
imb_targets = [train_ds.targets[i] for i in keep]
print(pd.Series(imb_targets).value_counts().sort_index())

pd.DataFrame({'index': keep, 'path': [train_ds.samples[i][0] for i in keep]}) \
  .to_csv('../data/processed/imbalanced_train_indices.csv', index=False)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
{'ambulance': 0, 'autobus': 1, 'kamyun': 2, 'kamyunet': 3, 'minibus': 4, 'savari': 5, 'taxi': 6, 'vanet': 7}
0     35
1    196
2    200
3    191
4     54
5    198
6    199
7    199
Name: count, dtype: int64


In [7]:
sampler = BalancedSampler(imb_targets , 32 , num_batches=10 , seed=SEED)
batch=next(iter(sampler))
print(collections.Counter(imb_targets[i] for i in batch))

Counter({1: 4, 6: 4, 5: 4, 2: 4, 0: 4, 4: 4, 7: 4, 3: 4})


In [10]:

run_experiment('imb_standard', train_subset=keep, sampler_type='none')

run_experiment('imb_balanced', train_subset=keep, sampler_type='balanced')

NameError: name 'run_experiment' is not defined